# Diabetes — XAI extension

The reusable XAI framework is applied to the pre-specified Random Forest with 300 trees using Glucose, BMI, Age, Insulin and SkinThickness. Median imputation is fitted on the training data, with 230 observations retained for held-out evaluation.

The framework generates explanations and assesses bootstrap-refit stability. Global permutation importance uses ROC AUC consistently across all five configurations, while bootstrap-refit stability uses negative log loss consistently across the four retrainable predictive configurations. Results are exported to `../../results/diabetes`.



The exported global SHAP analysis uses SHAP 0.51.0 `TreeExplainer`.


## Data and model preparation


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "xai_extension":
    raise RuntimeError(
        "Run this notebook with the working directory set to "
        "thesis/src/xai_extension. Current directory: "
        f"{NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
FRAMEWORK_DIR = PROJECT_ROOT / "src" / "xai_framework"
if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline

data_path = "../../data/diabetes/diabetes.csv"
df = pd.read_csv(data_path)
zero_invalid = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]


c:\Users\catia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
full_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
             "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]

cols_3 = ["Glucose", "BMI", "Age"]
cols_5 = ["Glucose", "BMI", "Age", "Insulin", "SkinThickness"]


In [3]:
# Predictive branch: split before learning median-imputation values.
X = df[full_cols].astype(float).copy()
X[zero_invalid] = X[zero_invalid].replace(0, np.nan)
y = df["Outcome"].copy()

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X, y, test_size=230, stratify=y, random_state=42
)

predictive_imputer = SimpleImputer(strategy="median")
X_train_full.loc[:, zero_invalid] = predictive_imputer.fit_transform(
    X_train_full[zero_invalid]
)
X_test_full.loc[:, zero_invalid] = predictive_imputer.transform(
    X_test_full[zero_invalid]
)

X_train_3 = X_train_full[cols_3]
X_test_3 = X_test_full[cols_3]

X_train_5 = X_train_full[cols_5]
X_test_5 = X_test_full[cols_5]

print(X_train_full.shape, X_test_full.shape)
print("\nTrain class counts:")
print(y_train.value_counts())
print("\nTest class counts:")
print(y_test.value_counts())


(538, 8) (230, 8)

Train class counts:
Outcome
0    350
1    188
Name: count, dtype: int64

Test class counts:
Outcome
0    150
1     80
Name: count, dtype: int64


In [4]:
dataset_map = {
    "Full (8 features)": (X_train_full, X_test_full),
    "3-factor": (X_train_3, X_test_3),
    "5-factor": (X_train_5, X_test_5)
}


In [5]:
def create_model(seed=42):
    """Create the pre-specified random-forest estimator."""
    if best_model_name != "Random Forest":
        raise ValueError(f"This extension is pre-specified for Random Forest: {best_model_name}")
    return RandomForestClassifier(n_estimators=300, random_state=seed)


In [6]:
# Fit the pre-specified Diabetes model.

# Pre-specified extension configuration; held-out test metrics above are
# reported descriptively and are not used for model selection.
best_model_name = "Random Forest"
best_dataset_name = "5-factor"
print("Pre-specified XAI model:", best_model_name)
print("Pre-specified feature set:", best_dataset_name)

X_train_xai, X_test_xai = dataset_map[best_dataset_name]

best_model = create_model(seed=42)
best_model.fit(X_train_xai, y_train)


def train_model_fn(X_bootstrap, y_bootstrap):
    """Refit the fixed Diabetes estimator on one bootstrap training sample."""
    model = create_model(seed=42)
    model.fit(X_bootstrap, y_bootstrap)
    return model


Pre-specified XAI model: Random Forest
Pre-specified feature set: 5-factor


## Execute the XAI pipeline


In [7]:
PILOT = False

config = {
    "dataset_name": "diabetes",

    "model": best_model,

    "X_train": X_train_xai,
    "X_test": X_test_xai,

    "y_train": y_train,
    "y_test": y_test,

    "feature_names": list(X_train_xai.columns),

    # Repeated local explanations
    "local_cases_per_type": 2 if PILOT else 10,
    "lime_seeds": list(range(2 if PILOT else 20)),
    "lime_num_features": 5,
    "lime_stability_top_k": 3,

    "train_model_fn": train_model_fn,

    # Bootstrap-refit stability
    "stability_n_bootstrap": 10 if PILOT else 100,
    "stability_bootstrap_seed": 42,
    "stability_scoring": "neg_log_loss",
    "stability_n_repeats": 10,
    "stability_permutation_seed": 42,
    "stability_top_k": 3,

    # Ordinary global permutation analysis
    "scoring": "roc_auc",

    # Keep pilot and final results separate
    "output_dir": (
        "../../results/pilot_stability/diabetes"
        if PILOT
        else "../../results/diabetes"
    ),
}

experiment = run_pipeline(config)


Results exported to: ../../results/diabetes


## XAI framework output sanity checks


In [8]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))


Dataset: diabetes
Model: RandomForestClassifier
X_train shape: (538, 5)
X_test shape: (230, 5)
y_train length: 538
y_test length: 230
Feature names: 5


### Input consistency


In [9]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("Basic shape validation: OK")


Basic shape validation: OK


### Prediction-interface check


In [10]:
model = config["model"]
X_test = config["X_test"]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)[:, 1]
    print("Probability range:", probs.min(), probs.max())

preds = model.predict(X_test)

print("Predictions:", np.unique(preds, return_counts=True))
print("Prediction test: OK")


Probability range: 0.0 0.98
Predictions: (array([0, 1]), array([157,  73]))
Prediction test: OK


### Framework results


In [11]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print(global_results.keys())
print(local_results.keys())

if stability_results is not None:
    print(stability_results.keys())
else:
    print("Stability: not available")


dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'shap_explainer_class', 'rank_compare', 'rank_agreement'])
dict_keys(['cases', 'lime', 'lime_summary', 'lime_seed_stability'])
dict_keys(['raw', 'summary', 'rank_stability'])


### Global explanations


In [12]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("Native:")
display(native_df.head())

print("Permutation:")
display(perm_df.head())

print("SHAP:")
display(shap_df.head())


Native:


,Feature,Importance,Method
0,Glucose,0.353578,tree_importance
1,BMI,0.245440,tree_importance
2,Age,0.183218,tree_importance
3,Insulin,0.118572,tree_importance
4,SkinThickness,0.099192,tree_importance


Permutation:


,Feature,Importance,Std
0,Glucose,0.120367,0.029588
1,BMI,0.071819,0.021583
2,Age,0.026760,0.012626
3,Insulin,0.017025,0.011851
4,SkinThickness,0.014567,0.005718


SHAP:


,Feature,MeanAbsSHAP,MeanSHAP
0,Glucose,0.145322,0.001546
1,BMI,0.087032,-0.013312
2,Age,0.069504,0.001925
3,Insulin,0.037430,-0.003399
4,SkinThickness,0.022289,0.003919


### Feature coverage


In [13]:
if native_df is not None:
    print("Native features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", len(config["feature_names"]))


Native features: 5
Permutation features: 5
SHAP features: 5
Expected features: 5


### Numerical integrity


In [14]:
print("Permutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())


Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0


### Rank agreement


In [15]:
print(global_results["rank_agreement"])

display(global_results["rank_compare"].head())


{'Native_vs_SHAP': np.float64(0.9999999999999999), 'Native_vs_Permutation': np.float64(0.9999999999999999), 'SHAP_vs_Permutation': np.float64(0.9999999999999999)}


,Feature,Importance_x,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,Glucose,0.353578,tree_importance,1.0,0.145322,0.001546,1.0,0.120367,0.029588,1.0
1,BMI,0.245440,tree_importance,2.0,0.087032,-0.013312,2.0,0.071819,0.021583,2.0
2,Age,0.183218,tree_importance,3.0,0.069504,0.001925,3.0,0.026760,0.012626,3.0
3,Insulin,0.118572,tree_importance,4.0,0.037430,-0.003399,4.0,0.017025,0.011851,4.0
4,SkinThickness,0.099192,tree_importance,5.0,0.022289,0.003919,5.0,0.014567,0.005718,5.0


### Representative cases


In [16]:
cases = local_results["cases"]

display(
    cases[
        ["CaseType", "y_true", "y_pred", "y_prob"]
    ]
)


,CaseType,y_true,y_pred,y_prob
52,TP,1,1,0.980000
138,TP,1,1,0.966667
92,TP,1,1,0.940000
82,TP,1,1,0.913333
143,TP,1,1,0.903333
14,TP,1,1,0.896667
175,TP,1,1,0.890000
31,TP,1,1,0.883333
110,TP,1,1,0.876667
196,TP,1,1,0.876667


### Local explanations and seed stability


In [17]:
lime_df = local_results["lime"]
lime_summary_df = local_results["lime_summary"]
lime_seed_stability_df = local_results["lime_seed_stability"]

print("Repeated LIME rows:", len(lime_df))
print("Cases by category:")
display(local_results["cases"].groupby("CaseType").size().rename("N cases"))
print("Seeds per case:")
display(lime_df.groupby(["CaseType", "CaseIndex"])["LIMESeed"].nunique().describe())
print("Aggregated LIME summary:")
display(lime_summary_df.head(20))
print("LIME seed-stability summary:")
display(
    lime_seed_stability_df.groupby("CaseType")[[
        "MeanPairwiseSpearman", "MeanTopKOverlap",
        "MeanRuleWeightSD", "MeanRuleSignConsistency"
    ]].mean().round(3)
)


Repeated LIME rows: 5000
Cases by category:


CaseType
Borderline    10
FN            10
FP            10
TN            10
TP            10
Name: N cases, dtype: int64

Seeds per case:


count    50.0
mean     20.0
std       0.0
min      20.0
25%      20.0
50%      20.0
75%      20.0
max      20.0
Name: LIMESeed, dtype: float64

Aggregated LIME summary:


,CaseType,Rule,MeanWeight,StdWeight,MeanAbsWeight,MedianWeight,PositiveWeightFraction,NExplanations,NCases,NSeeds,SelectionFrequency
0,Borderline,Glucose > 141.00,0.369874,0.007839,0.369874,0.369910,1.0,80,4,20,0.4
1,Borderline,BMI <= 27.82,-0.197630,0.004434,0.197630,-0.197864,0.0,20,1,20,0.1
2,Borderline,BMI > 36.38,0.139736,0.008900,0.139736,0.139653,1.0,60,3,20,0.3
3,Borderline,Insulin <= 122.00,-0.132366,0.007017,0.132366,-0.132894,0.0,40,2,20,0.2
4,Borderline,99.25 < Glucose <= 117.00,-0.121703,0.007042,0.121703,-0.122303,0.0,20,1,20,0.1
5,Borderline,Age > 41.00,0.112524,0.006286,0.112524,0.113042,1.0,80,4,20,0.4
6,Borderline,Insulin > 130.00,0.062213,0.007448,0.062213,0.061736,1.0,20,1,20,0.1
7,Borderline,122.00 < Insulin <= 126.00,0.048612,0.005513,0.048612,0.048697,1.0,120,6,20,0.6
8,Borderline,30.00 < Age <= 41.00,0.047921,0.008659,0.047921,0.048067,1.0,40,2,20,0.2
9,Borderline,SkinThickness > 32.00,0.045991,0.006695,0.045991,0.046025,1.0,40,2,20,0.2


LIME seed-stability summary:


,MeanPairwiseSpearman,MeanTopKOverlap,MeanRuleWeightSD,MeanRuleSignConsistency
CaseType,,,,
Borderline,0.952,0.912,0.007,1.000
FN,0.932,0.872,0.007,0.999
FP,0.945,0.945,0.007,1.000
TN,0.986,0.937,0.006,1.000
TP,0.965,0.963,0.006,1.000


### Bootstrap-refit stability


In [18]:
if stability_results is not None:
    display(stability_results["raw"].head())
    print(stability_results["summary"])


,Feature,Importance,ImportanceStd,Bootstrap,BootstrapSampleSeed,Rank,InTopK
0,Glucose,0.160955,0.083533,1,383329927,1.0,True
1,BMI,0.097812,0.059533,1,383329927,2.0,True
2,Age,0.022384,0.017022,1,383329927,4.0,False
3,Insulin,0.039501,0.050120,1,383329927,3.0,True
4,SkinThickness,0.006234,0.005959,1,383329927,5.0,False


         Feature  MeanImportance  StdImportance  MeanRank   StdRank  \
2        Glucose        0.160143       0.031602      1.01  0.100000   
1            BMI        0.086600       0.027565      2.10  0.333333   
0            Age        0.042100       0.033566      3.17  0.711450   
3        Insulin        0.014266       0.013210      4.31  0.662029   
4  SkinThickness        0.014592       0.021492      4.41  0.668105   

   TopKFrequency  
2           1.00  
1           1.00  
0           0.81  
3           0.11  
4           0.08  


### Exported artefacts


In [19]:

out_dir = Path(config["output_dir"]).resolve()

print("Output directory:", out_dir)

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))


Output directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\diabetes
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
execution_times.json
lime_explanations.csv
lime_seed_stability.csv
lime_summary.csv
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv
stability.csv
stability_rank.csv
stability_summary.csv


In [20]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)


['Glucose', 'BMI', 'Age', 'Insulin', 'SkinThickness', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex', 'CaseRank']


,Glucose,BMI,Age,Insulin,SkinThickness,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
52,171.0,43.6,26.0,126.0,29.0,1,1,0.980000,0.480000,TP,52,1
138,167.0,32.3,30.0,126.0,29.0,1,1,0.966667,0.466667,TP,138,2
92,196.0,36.5,29.0,249.0,36.0,1,1,0.940000,0.440000,TP,92,3
82,194.0,35.9,41.0,126.0,28.0,1,1,0.913333,0.413333,TP,82,4
143,173.0,35.7,22.0,474.0,33.0,1,1,0.903333,0.403333,TP,143,5
14,184.0,37.0,31.0,277.0,39.0,1,1,0.896667,0.396667,TP,14,6
175,164.0,30.8,32.0,126.0,21.0,1,1,0.890000,0.390000,TP,175,7
31,170.0,44.0,43.0,126.0,31.0,1,1,0.883333,0.383333,TP,31,8
110,143.0,36.6,51.0,146.0,33.0,1,1,0.876667,0.376667,TP,110,9
196,173.0,36.8,38.0,126.0,29.0,1,1,0.876667,0.376667,TP,196,10


In [21]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns
print("Diabetes CaseIndex exists in memory.")


Diabetes CaseIndex exists in memory.


In [22]:
diabetes_cases = pd.read_csv(
    Path("../../results/diabetes/representative_cases.csv")
)

print(diabetes_cases.columns.tolist())
display(diabetes_cases)


['Glucose', 'BMI', 'Age', 'Insulin', 'SkinThickness', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex', 'CaseRank']


,Glucose,BMI,Age,Insulin,SkinThickness,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
0,171.0,43.6,26.0,126.0,29.0,1,1,0.980000,0.480000,TP,52,1
1,167.0,32.3,30.0,126.0,29.0,1,1,0.966667,0.466667,TP,138,2
2,196.0,36.5,29.0,249.0,36.0,1,1,0.940000,0.440000,TP,92,3
3,194.0,35.9,41.0,126.0,28.0,1,1,0.913333,0.413333,TP,82,4
4,173.0,35.7,22.0,474.0,33.0,1,1,0.903333,0.403333,TP,143,5
5,184.0,37.0,31.0,277.0,39.0,1,1,0.896667,0.396667,TP,14,6
6,164.0,30.8,32.0,126.0,21.0,1,1,0.890000,0.390000,TP,175,7
7,170.0,44.0,43.0,126.0,31.0,1,1,0.883333,0.383333,TP,31,8
8,143.0,36.6,51.0,146.0,33.0,1,1,0.876667,0.376667,TP,110,9
9,173.0,36.8,38.0,126.0,29.0,1,1,0.876667,0.376667,TP,196,10


In [23]:

pilot_dir = Path("../../results/pilot_stability/diabetes").resolve()
stability = pd.read_csv(pilot_dir / "stability.csv")

importance_by_bootstrap = stability.pivot(
    index="Bootstrap",
    columns="Feature",
    values="Importance",
)

print("Bootstrap refits:", len(importance_by_bootstrap))
print(
    "Distinct importance profiles:",
    len(importance_by_bootstrap.drop_duplicates())
)

display(importance_by_bootstrap)


Bootstrap refits: 10
Distinct importance profiles: 10


Feature,Age,BMI,Glucose,Insulin,SkinThickness
Bootstrap,,,,,
1,0.022384,0.097812,0.160955,0.039501,0.006234
2,0.052068,0.082797,0.206452,0.014826,0.003771
3,0.088187,0.108281,0.154673,0.013154,0.020232
4,0.068394,0.081552,0.135772,0.031843,0.013950
5,0.049295,0.050800,0.154731,-0.009301,0.003965
6,0.023269,0.095602,0.133163,0.011911,-0.000711
7,0.043386,0.085601,0.128192,0.009853,0.014475
8,0.041039,0.084458,0.156608,0.010138,0.014907
9,0.022808,0.102329,0.161931,0.022098,0.005829


In [24]:

results_dir = Path(config["output_dir"]).resolve()
stability_file = results_dir / "stability.csv"

print("Reading:", stability_file)

stability = pd.read_csv(stability_file)

importance_by_bootstrap = stability.pivot(
    index="Bootstrap",
    columns="Feature",
    values="Importance",
)

print("Bootstrap refits:", len(importance_by_bootstrap))
print(
    "Distinct importance profiles:",
    len(importance_by_bootstrap.drop_duplicates()),
)

display(importance_by_bootstrap)


Reading: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\diabetes\stability.csv
Bootstrap refits: 100
Distinct importance profiles: 100


Feature,Age,BMI,Glucose,Insulin,SkinThickness
Bootstrap,,,,,
1,0.022384,0.097812,0.160955,0.039501,0.006234
2,0.052068,0.082797,0.206452,0.014826,0.003771
3,0.088187,0.108281,0.154673,0.013154,0.020232
4,0.068394,0.081552,0.135772,0.031843,0.013950
5,0.049295,0.050800,0.154731,-0.009301,0.003965
...,...,...,...,...,...
96,0.017703,0.075459,0.147870,0.014071,0.015539
97,0.083575,0.067676,0.141972,0.002449,0.019055
98,0.053848,0.106980,0.175051,0.014842,0.003015


## Interpretation

The three global methods produced the same feature ordering in this five-feature configuration. Glucose and BMI were also the most persistent leading features across refits, and the configuration had the highest top-three overlap. Because the feature space is small, complete or near-complete top-feature overlap is easier to obtain and should not be interpreted as universal agreement.
